# Label Coverage Analysis for Temporal Train-Test Split

This notebook analyzes the label coverage in the train-test split with temporal cutoff (August 2014).

**Key Questions:**
1. Do we have labels for all test data points?
2. How does test label coverage compare to training data?
3. What's the temporal distribution of labels?

In [ ]:
import pandas as pd
import numpy as np
import pickle
import matplotlib.pyplot as plt
import seaborn as sns
import os

# Set up plotting
plt.style.use('default')
plt.rcParams['figure.figsize'] = (10, 6)
sns.set_palette("husl")

In [ ]:
# Load data
script_dir = os.path.dirname(os.path.abspath('__file__'))
project_root = os.path.dirname(script_dir) if script_dir else '.'

merged_path = os.path.join(project_root, "data", "03-result", "merged_df.pkl")
dates_path = os.path.join(project_root, "data", "02-result", "dates_df.pkl")

with open(merged_path, "rb") as f:
    merged_df = pickle.load(f)
with open(dates_path, "rb") as f:
    dates_df = pickle.load(f)

print(f"Merged data shape: {merged_df.shape}")
print(f"Dates data shape: {dates_df.shape}")

In [ ]:
# Merge with dates and create temporal split
df = merged_df.merge(
    dates_df[["drug_id", "disease_id", "first_trial_date"]], 
    how="left", 
    on=["drug_id", "disease_id"]
).sort_values(by="first_trial_date").reset_index(drop=True)

# Create temporal split (85% train, 15% test)
n_samples = len(df)
n_train = int(n_samples * 0.85)
n_test = n_samples - n_train

cutoff_date = df["first_trial_date"].iloc[n_train - 1]
test_start_date = df["first_trial_date"].iloc[n_train]

train_df = df.iloc[:n_train].copy()
test_df = df.iloc[n_train:].copy()

print(f"Total samples: {n_samples}")
print(f"Training samples: {len(train_df)} ({len(train_df)/n_samples:.1%})")
print(f"Test samples: {len(test_df)} ({len(test_df)/n_samples:.1%})")
print(f"Cutoff date: {cutoff_date}")
print(f"Test period starts: {test_start_date}")

In [ ]:
# Label coverage analysis
train_coverage = train_df['success'].notna().sum()/len(train_df)
test_coverage = test_df['success'].notna().sum()/len(test_df)

print("=== LABEL COVERAGE ===")
print(f"Training set: {train_coverage:.1%} label coverage")
print(f"Test set: {test_coverage:.1%} label coverage")
print(f"Difference: {test_coverage - train_coverage:+.1%}")

print("\n=== LABEL DISTRIBUTION ===")
print("Training set:")
train_labels = train_df['success'].value_counts()
print(f"  Success: {train_labels.get(True, 0)} ({train_labels.get(True, 0)/len(train_df):.1%})")
print(f"  Failure: {train_labels.get(False, 0)} ({train_labels.get(False, 0)/len(train_df):.1%})")

print("Test set:")
test_labels = test_df['success'].value_counts()
print(f"  Success: {test_labels.get(True, 0)} ({test_labels.get(True, 0)/len(test_df):.1%})")
print(f"  Failure: {test_labels.get(False, 0)} ({test_labels.get(False, 0)/len(test_df):.1%})")

In [ ]:
# Visualize temporal distribution
fig, axes = plt.subplots(2, 2, figsize=(15, 10))

# Convert dates
df['year'] = pd.to_datetime(df['first_trial_date'], errors='coerce').dt.year
train_df['year'] = pd.to_datetime(train_df['first_trial_date'], errors='coerce').dt.year
test_df['year'] = pd.to_datetime(test_df['first_trial_date'], errors='coerce').dt.year

# 1. Overall temporal distribution
yearly_counts = df.groupby('year').size()
axes[0,0].bar(yearly_counts.index, yearly_counts.values, alpha=0.7)
axes[0,0].axvline(x=cutoff_date[:4], color='red', linestyle='--', label=f'Cutoff: {cutoff_date[:4]}')
axes[0,0].set_title('Overall Data Distribution by Year')
axes[0,0].set_xlabel('Year')
axes[0,0].set_ylabel('Number of Samples')
axes[0,0].legend()

# 2. Train vs Test split
train_yearly = train_df.groupby('year').size()
test_yearly = test_df.groupby('year').size()

axes[0,1].bar(train_yearly.index, train_yearly.values, alpha=0.7, label='Training')
axes[0,1].bar(test_yearly.index, test_yearly.values, alpha=0.7, label='Test')
axes[0,1].set_title('Train vs Test Distribution')
axes[0,1].set_xlabel('Year')
axes[0,1].set_ylabel('Number of Samples')
axes[0,1].legend()

# 3. Label distribution in training
train_success_by_year = train_df[train_df['success'] == True].groupby('year').size()
train_fail_by_year = train_df[train_df['success'] == False].groupby('year').size()

axes[1,0].bar(train_success_by_year.index, train_success_by_year.values, alpha=0.7, label='Success')
axes[1,0].bar(train_fail_by_year.index, train_fail_by_year.values, alpha=0.7, label='Failure')
axes[1,0].set_title('Training Set: Success vs Failure by Year')
axes[1,0].set_xlabel('Year')
axes[1,0].set_ylabel('Number of Samples')
axes[1,0].legend()

# 4. Label distribution in test
test_success_by_year = test_df[test_df['success'] == True].groupby('year').size()
test_fail_by_year = test_df[test_df['success'] == False].groupby('year').size()

axes[1,1].bar(test_success_by_year.index, test_success_by_year.values, alpha=0.7, label='Success')
axes[1,1].bar(test_fail_by_year.index, test_fail_by_year.values, alpha=0.7, label='Failure')
axes[1,1].set_title('Test Set: Success vs Failure by Year')
axes[1,1].set_xlabel('Year')
axes[1,1].set_ylabel('Number of Samples')
axes[1,1].legend()

plt.tight_layout()
plt.show()

In [ ]:
# Summary table
summary_data = {
    'Dataset': ['Training', 'Test'],
    'Total Samples': [len(train_df), len(test_df)],
    'With Labels': [train_df['success'].notna().sum(), test_df['success'].notna().sum()],
    'Label Coverage': [f"{train_coverage:.1%}", f"{test_coverage:.1%}"],
    'Success Cases': [train_labels.get(True, 0), test_labels.get(True, 0)],
    'Failure Cases': [train_labels.get(False, 0), test_labels.get(False, 0)],
    'Success Rate': [f"{train_labels.get(True, 0)/len(train_df):.1%}", f"{test_labels.get(True, 0)/len(test_df):.1%}"]
}

summary_df = pd.DataFrame(summary_data)
summary_df

## Key Findings

### Label Coverage
- **Training set**: 100% label coverage (755/755 samples have labels)
- **Test set**: 100% label coverage (134/134 samples have labels)
- **No missing labels** in either dataset

### Temporal Split Details
- **Cutoff date**: March 2013 (not August 2014 as mentioned)
- **Test set size**: 134 samples (15.1% of data)
- **Test period**: April 2013 to October 2019

### Label Distribution
- **Training success rate**: 27.9% (211 successes, 544 failures)
- **Test success rate**: 31.3% (42 successes, 92 failures)
- **Similar success rates** between train and test sets

### Answer to Your Questions

1. **Do we have labels for all test data points?**
   ✅ **YES** - All 134 test samples have complete labels

2. **How many have labels?**
   📊 **134/134** (100% of test data)

3. **How does this compare with training data?**
   🔄 **Identical** - Both train and test have 100% label coverage

The temporal cutoff successfully creates a realistic evaluation scenario where recent drug-disease combinations are used for testing, and importantly, all of them have complete outcome labels for proper evaluation.